# Módulo 2 — Primer RAG completo, sin framework


In [ ]:
!pip install sentence-transformers -q


In [ ]:
from pathlib import Path
from sentence_transformers import SentenceTransformer, util
import numpy as np, re, requests, os, json

# Descarga del corpus desde GitHub si estás en Colab.
# Editá OWNER y REPO si cambiaste el nombre propuesto.
OWNER = "TU_USUARIO"
REPO = "curso-rag-acmecloud"
BASE = f"https://raw.githubusercontent.com/{OWNER}/{REPO}/main/corpus"

for nombre in ["catalogo_sistemas.txt","politicas_it.txt","manual_soporte.txt"]:
    r = requests.get(f"{BASE}/{nombre}", timeout=30)
    if r.status_code == 200:
        Path(f"/content/{nombre}").write_text(r.text, encoding="utf-8")
    else:
        print("No pude descargar", nombre, "- subilo manualmente o configurá OWNER/REPO.")


In [ ]:
from sentence_transformers import SentenceTransformer, util
modelo = SentenceTransformer("paraphrase-multilingual-MiniLM-L12-v2")

archivos = ["/content/catalogo_sistemas.txt","/content/politicas_it.txt","/content/manual_soporte.txt"]
documentos = {}
for ruta in archivos:
    with open(ruta, encoding="utf-8") as f:
        documentos[Path(ruta).name] = f.read()

def trocear_fichas(texto):
    partes = texto.split("###")
    return ["###"+p.strip() for p in partes[1:] if p.strip()]

def trocear_prosa(texto, min_car=200):
    parrafos = [p.strip() for p in texto.split("\n\n") if p.strip()]
    trozos, buffer = [], ""
    for p in parrafos:
        buffer = (buffer+"\n\n"+p).strip()
        if len(buffer) >= min_car:
            trozos.append(buffer); buffer=""
    if buffer: trozos.append(buffer)
    return trozos

chunks, origen = [], []
for nombre,texto in documentos.items():
    piezas = trocear_fichas(texto) if "catalogo" in nombre else trocear_prosa(texto)
    for p in piezas:
        chunks.append(p); origen.append(nombre)

vectores = modelo.encode(chunks, show_progress_bar=True)
print("Chunks:", len(chunks), "Dimensiones:", vectores.shape[1])


In [ ]:
def buscar(pregunta, k=4):
    vq = modelo.encode(pregunta)
    scores = util.cos_sim(vq, vectores)[0].numpy()
    idxs = np.argsort(-scores)[:k]
    return [(int(i), float(scores[i])) for i in idxs]

for idx,score in buscar("¿Qué significa VPN-1047?"):
    print(round(score,3), origen[idx], chunks[idx][:250].replace("\n"," "))


Probá también una pregunta agregada como **¿Cuántos sistemas usan MFA?**. El objetivo es observar por qué un top-k no puede contar el corpus completo.
